# Clamp Detection — YOLO Training

Colab-friendly notebook that uses [`train_yolo.py`](https://github.com/BASSAT-BASSAT/Clamp-Detection) from this repo (no duplicated training code).

| Step | What it does |
|------|----------------|
| 1 | Install dependencies |
| 2 | Clone repo & import `train_yolo` |
| 3 | Configure paths & hyperparameters |
| 4 | Prepare dataset (**80% / 15% / 5%** train/val/test) |
| 5 | Train YOLO11 |
| 6 | Run inference on a video |

**Dataset:** download from [Google Drive](https://drive.google.com/drive/folders/1-2gRkTrO4aAEHxRB9ziJwSFd5doToDPl?usp=sharing) and place files next to the paths in the config cell.

In [ ]:
%pip install -q ultralytics torch opencv-python

## 1. Get the code

Clones the GitHub repo on Colab. When running **locally** inside this project folder, the clone step is skipped automatically.

In [ ]:
import shutil
import sys
from pathlib import Path

REPO_URL = "https://github.com/BASSAT-BASSAT/Clamp-Detection.git"
REPO_DIR = Path("/content/Clamp-Detection")  # Colab default

# Use current folder if train_yolo.py already exists (local run)
if (Path.cwd() / "train_yolo.py").exists():
    REPO_DIR = Path.cwd()
    print(f"Using local repo: {REPO_DIR.resolve()}")
elif not (REPO_DIR / "train_yolo.py").exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")
else:
    print(f"Using existing clone: {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from train_yolo import prepare_yolo_dataset, train_model, predict_video

## 2. Configuration

Edit paths for your environment. Colab example paths are shown below.

In [ ]:
#  Paths
DATASET_ROOT = Path("/content/drive/MyDrive/Clamp Dataset/Clamp Detection.coco")
OUTPUT_ROOT = Path("/content/drive/MyDrive/Clamp Dataset/clamp_detection_yolo")
INPUT_VIDEO = Path("/content/drive/MyDrive/Clamp Dataset/Task.mp4")

# Local example (uncomment if not using Colab):
# DATASET_ROOT = REPO_DIR / "Clamp Detection.coco"
# OUTPUT_ROOT = REPO_DIR / "clamp_detection_yolo"
# INPUT_VIDEO = REPO_DIR / "Task.mp4"

#  Training 
MODEL = "yolo11m.pt"
EPOCHS = 50
IMGSZ = 640
SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.80, 0.15, 0.05

PROJECT_DIR = REPO_DIR / "runs"
RUN_NAME = "clamp_detection"
CONF_THRESHOLD = 0.25

## 3. Prepare dataset

In [ ]:
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

data_yaml = prepare_yolo_dataset(
    dataset_root=DATASET_ROOT,
    output_root=OUTPUT_ROOT,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
)
print(f"Dataset ready: {OUTPUT_ROOT.resolve()}")
print(f"Config: {data_yaml}")

## 4. Train

In [ ]:
best_weights = train_model(
    data_yaml=data_yaml,
    model_name=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    project_dir=PROJECT_DIR,
    run_name=RUN_NAME,
)
print(f"Best weights: {best_weights}")

## 5. Test on input video

In [ ]:
output_video = predict_video(
    input_video=INPUT_VIDEO,
    project_dir=PROJECT_DIR,
    run_name=RUN_NAME,
    conf=CONF_THRESHOLD,
    imgsz=IMGSZ,
)

if output_video.suffix in {".mp4", ".avi", ".mov"}:
    try:
        from IPython.display import Video, display

        display(Video(str(output_video), embed=True, width=960))
    except Exception as error:
        print(f"Preview skipped: {error}")